# PRÁCTICA BACKTESTING AVANZADO

##### Realizado por Mateo Santos

In [1]:
# Importamos las librerías
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import pyarrow
import pyarrow.parquet as pq
import time

# Fijamos la semilla
np.random.seed(42)

In [2]:
# Definición de variables globales
capital = 250000
#START_DATE = '2015-01-01'
START_DATE = '2014-01-01'
END_DATE = '2026-01-31'
NUM_ACTIVOS = 20
capital_por_activo = capital/NUM_ACTIVOS

## Notebook 4: Ejecución y costes

Filtramos los precios de compra y venta de cada una de las fechas de rebalanceo en las variables `precios_compra` y `precios_venta`.

In [30]:
# Filtramos los precios de compra (close) y venta (open) en función de las fechas en las que se realizan los rebalanceos
precios_compra = precios_close.loc[fechas_reales]
precios_compra.index = fechas_reales

precios_venta = precios_open.loc[fechas_reales]
precios_venta.index = fechas_reales

Con todo lo desarrollado anteriormente, tenemos tres variables clave:
- **`select_assets_period`**: Almacena la matriz de pesos en las que se determina, para cada periodo, en que activos hay que invertir
- **`precios_compra`**: Almacena los precios de compra (Close) sobre los que se hará el ajuste tras rebalanceo
- **`precios_venta`**: Almacena los precios de venta (Open) sobre los que se hará el ajuste tras rebalanceo

Además, a continuación se desarrolla la función que calcula las comisiones. En el caso en el que la cantidad sea cero, no hay comisión. En cualquier otro caso, la comisión se calcula como al máximo entre 23 y 0,23% multiplicado por el monto a invertir.

In [31]:
# Función para el cálculo de las comisiones de cada operación
def comision(cantidad, porc = 0.0023, minimo = 23):
    if cantidad == 0:
        comision = 0
    else:
        comision = np.max([np.abs(cantidad)*porc, minimo])
    return comision

El procedimiento para el cálculo del resultado de la estrategia se realiza mediante la clase **`BacktestEngine`**. Para instanciar la clase debemos incluir en el constructor las variables: 

- **capital**: Cantidad total que se va a invertir en cada periodo.
- **num_activos**: Número de activos en los que se va a invertir de forma equiponderada.
- **precios_close**: Precios sobre los que se compran las acciones tras rebalanceo.
- **precios_open**: Precios sobre los que se venden las acciones tras rebalanceo.
- **matriz_sp500**: Matriz booleana que indica para cada activo y fecha su presencia en el S&P 500.
- **select_assets_period**: Matriz booleana que indica para cada periodo, en que activos se va a invertir.
- **fechas_reales**: Fechas de rebalanceo, ajustadas sobre el último día hábil bursátil de cada mes.
- **func_comision**: Función que calcula la comisión de cada operación.

La clase cuenta con solo una función. La función **`run()`** se encarga de realizar todos los cálculos sobre el rendimiento de la estrategia Momentum que hemos desarrollado. Su funcionamiento se detalla a continuación:

1. Comenzamos con un diccionario vacío que almacenará para cada periodo los **tickers** de los activos en los que se invierte, que serán las claves y el **número de acciones invertidas**, que serán los valores.
2. En el **primer rebalanceo** (último día bursátil hábil de enero de 2015) **se comprarán las acciones** con su respectiva comisión.
3. **Durante cada uno de los meses**, si uno de los activos de la cartera ha salido del S&P 500 o ha dejado de cotizar, se vende en ese mismo día por su valor de cierre. **Ese efectivo se mantendrá parado hasta el siguiente rebalanceo**.
4. En el resto de casos el rebalanceo consistirá en:
    - **Vender o comprar acciones hasta que se llegue a la cantidad de 12.500 dólares** con su respectiva comisión. **No se pueden comprar ni vender acciones particionadas**.
5. En la variable **`equity_curve`** se guardan los resultados mensuales de la estrategia de inversión momentum.

In [32]:
class BacktestEngine:
    def __init__(self, capital, num_activos, precios_close, precios_open, 
                 matriz_sp500, select_assets_period, fechas_reales, func_comision):
        self.capital_inicial = capital
        self.num_activos = num_activos
        self.target_per_asset = capital / num_activos
        
        # Precios originales (sin ffill masivo para detectar huecos reales)
        self.precios_close = precios_close
        self.precios_open = precios_open
        
        self.matriz_sp500 = matriz_sp500
        self.select_assets_period = select_assets_period
        self.fechas_reales = fechas_reales
        self.comision = func_comision
        
        # Metadata de supervivencia
        self.ultimo_dia_cotizacion = precios_close.notna()[::-1].idxmax()
        self.todos_los_dias = precios_close.index

    def run(self):
        cash = self.capital_inicial
        cartera_actual = {} 
        valor_total_anterior = self.capital_inicial
        
        hist = {'fechas': [], 'valor_total': [], 'cash': [], 'comisiones': [], 'pnl': []}

        for fecha in self.todos_los_dias:
            comisiones_dia = 0.0
            
            # 1) CONTROL DIARIO: CIERRES FORZADOS (Delisting o salida SP500)
            for activo in list(cartera_actual.keys()):
                esta_en_sp = self.matriz_sp500.loc[fecha, activo] if activo in self.matriz_sp500.columns else False
                sigue_cotizando = fecha < self.ultimo_dia_cotizacion[activo]

                if not esta_en_sp or not sigue_cotizando:
                    # Intentamos obtener precio real, si no, el último disponible
                    p_salida = self.precios_close.loc[fecha, activo]
                    if pd.isna(p_salida):
                        p_salida = self.precios_close[activo].loc[:fecha].ffill().iloc[-1]
                    
                    valor_venta = cartera_actual[activo] * p_salida
                    com = self.comision(valor_venta)
                    
                    cash += valor_venta - com
                    comisiones_dia += com
                    
                    motivo = "Salida SP500" if not esta_en_sp else "Delisting/Fin Cotización"
                    print(f"[{fecha.date()}] Venta forzosa: {activo} por {motivo}")
                    del cartera_actual[activo]

            # 2) REBALANCEO MENSUAL
            if fecha in self.fechas_reales:
                # Obtenemos candidatos del ranking
                candidatos = self.select_assets_period.loc[fecha]
                activos_objetivo_bruto = candidatos[candidatos > 0].index
                
                # --- FILTRO DE SEGURIDAD ANTIZOMBI ---
                # Solo consideramos activos que SIGAN en el SP500 y COTICEN hoy
                activos_objetivo = [
                    a for a in activos_objetivo_bruto 
                    if (fecha < self.ultimo_dia_cotizacion[a]) and 
                       (self.matriz_sp500.loc[fecha, a] if a in self.matriz_sp500.columns else False)
                ]

                # A. VENTAS (Usamos precio Open de hoy)
                for activo in list(cartera_actual.keys()):
                    p_venta = self.precios_open.loc[fecha, activo]
                    # Si el Open es NaN, buscamos el último cierre conocido
                    if pd.isna(p_venta):
                        p_venta = self.precios_close[activo].loc[:fecha].ffill().iloc[-1]
                        
                    valor_actual = cartera_actual[activo] * p_venta

                    if activo not in activos_objetivo:
                        com = self.comision(valor_actual)
                        cash += valor_actual - com
                        comisiones_dia += com
                        del cartera_actual[activo]
                    elif valor_actual > self.target_per_asset:
                        n_mantener = int(np.floor(self.target_per_asset / p_venta))
                        n_vender = cartera_actual[activo] - n_mantener
                        if n_vender > 0:
                            monto_vender = n_vender * p_venta
                            com = self.comision(monto_vender)
                            cash += monto_vender - com
                            comisiones_dia += com
                            cartera_actual[activo] = n_mantener

                # B. COMPRAS (Usamos precio Close de hoy)
                for activo in activos_objetivo:
                    p_compra = self.precios_close.loc[fecha, activo]
                    # Si no hay precio de compra hoy, no podemos entrar
                    if pd.isna(p_compra):
                        continue
                        
                    n_actual = cartera_actual.get(activo, 0)
                    valor_actual = n_actual * p_compra
                    
                    if valor_actual < self.target_per_asset:
                        presupuesto = self.target_per_asset - valor_actual
                        n_comprar = int(np.floor(presupuesto / p_compra))
                        
                        if n_comprar > 0:
                            coste_est = n_comprar * p_compra
                            com = self.comision(coste_est)
                            if (coste_est + com) > presupuesto:
                                n_comprar -= 1
                                coste_est = n_comprar * p_compra
                                com = self.comision(coste_est)
                            
                            if n_comprar > 0 and cash >= (coste_est + com):
                                cash -= (coste_est + com)
                                comisiones_dia += com
                                cartera_actual[activo] = n_actual + n_comprar

            # 3) VALORACIÓN DIARIA ROBUSTA
            valor_acciones = 0.0
            for activo, cantidad in cartera_actual.items():
                precio_hoy = self.precios_close.loc[fecha, activo]
                if pd.isna(precio_hoy):
                    # Solo aquí aplicamos ffill puntual para no romper la valoración
                    precio_hoy = self.precios_close[activo].loc[:fecha].ffill().iloc[-1]
                valor_acciones += cantidad * (precio_hoy if pd.notna(precio_hoy) else 0.0)

            valor_total = cash + valor_acciones
            if pd.isna(valor_total): valor_total = valor_total_anterior

            hist['fechas'].append(fecha)
            hist['valor_total'].append(valor_total)
            hist['cash'].append(cash)
            hist['comisiones'].append(comisiones_dia)
            hist['pnl'].append(valor_total - valor_total_anterior)
            valor_total_anterior = valor_total

        return pd.DataFrame(hist).set_index('fechas')

In [33]:
backtesting = BacktestEngine(capital, NUM_ACTIVOS, precios_venta, 
                 precios_compra, matriz_sp500, select_assets_period, fechas_reales, comision)

In [34]:
backtesting_result = backtesting.run()

equity_curve = backtesting_result.iloc[:,0]
cash_series = backtesting_result.iloc[:,1]
comisiones_series = backtesting_result.iloc[:,2]
pnl_series = backtesting_result.iloc[:,-1]

[2015-02-27] Venta forzosa: AGN-201503 por Delisting/Fin Cotización
[2015-02-27] Venta forzosa: CFN-201503 por Delisting/Fin Cotización
[2015-06-30] Venta forzosa: KRFT-201507 por Delisting/Fin Cotización
[2015-07-31] Venta forzosa: PLL-201508 por Delisting/Fin Cotización
[2015-08-31] Venta forzosa: HSP-201509 por Delisting/Fin Cotización
[2015-11-30] Venta forzosa: ALTR-201512 por Delisting/Fin Cotización
[2015-12-31] Venta forzosa: CB-201601 por Delisting/Fin Cotización
[2016-03-31] Venta forzosa: CAM-201604 por Delisting/Fin Cotización
[2016-04-29] Venta forzosa: ARG-201605 por Delisting/Fin Cotización
[2016-06-30] Venta forzosa: TE-201606 por Delisting/Fin Cotización
[2017-01-31] Venta forzosa: SE-201702 por Delisting/Fin Cotización
[2018-08-31] Venta forzosa: XL-201809 por Delisting/Fin Cotización
[2018-09-28] Venta forzosa: ANDV-201809 por Delisting/Fin Cotización
[2019-08-30] Venta forzosa: TSS-201909 por Delisting/Fin Cotización
[2020-04-30] Venta forzosa: AGN-202005 por Delist

Guardamos los resultados mensuales acumulados de la estrategia momentum, para poder usarlo en el notebook de la simulación de Monte Carlo (Monos).

In [35]:
# Guardamos el resultado mensual de la estrategia para poder exportarlo al notebook de los monos
equity_curve.to_csv('equity_estrategia.csv')